# Video to Image

로컬 영상 파일(`mp4`, `gif`, `avi` 등)을 불러와서 지정한 간격마다 프레임 이미지를 저장합니다.

- `file_path`: 영상 파일 경로
- `frame_number = 1`: 모든 프레임 저장
- `frame_number = 5`: 5프레임마다 1장씩 저장
- 저장 폴더는 기본적으로 영상 파일 옆에 자동 생성됩니다.


In [1]:
from pathlib import Path

import imageio.v2 as imageio
import numpy as np
from PIL import Image

# =========================
# User settings
# =========================
file_path = r"C:\Users\user\Desktop\Gibeom\1. HI Lab\0. Projects\0. On going\1. [Lead] ionic DC-TENG\0. 실험자료\8. Gibeom touch\260604_지원 카톡 합본 (최종)\DC Final  movie.gif"

frame_number = 1  # 1이면 모든 프레임 저장, 5이면 5프레임마다 1장 저장
image_extension = "png"  # png 또는 jpg 추천
output_dir = None  # None이면 영상 파일 옆에 자동 생성

In [2]:



def to_uint8(frame: np.ndarray) -> np.ndarray:
    """Convert a frame to uint8 so it can be saved consistently."""
    frame = np.asarray(frame)

    if frame.dtype == np.uint8:
        return frame

    if np.issubdtype(frame.dtype, np.floating):
        frame = np.clip(frame, 0.0, 1.0) * 255.0
    else:
        frame = np.clip(frame, 0, 255)

    return frame.astype(np.uint8)


def prepare_image(frame: np.ndarray) -> Image.Image:
    """Handle grayscale, RGB, and RGBA frames."""
    frame = to_uint8(frame)

    if frame.ndim == 2:
        return Image.fromarray(frame, mode="L")

    if frame.ndim == 3 and frame.shape[2] == 1:
        return Image.fromarray(frame[:, :, 0], mode="L")

    if frame.ndim == 3 and frame.shape[2] == 3:
        return Image.fromarray(frame, mode="RGB")

    if frame.ndim == 3 and frame.shape[2] == 4:
        return Image.fromarray(frame, mode="RGBA")

    raise ValueError(f"지원하지 않는 프레임 shape입니다: {frame.shape}")


def save_video_frames(file_path: str, frame_number: int = 1, image_extension: str = "png", output_dir: str | None = None) -> None:
    video_path = Path(file_path)

    if not video_path.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {video_path}")

    if frame_number < 1:
        raise ValueError("frame_number는 1 이상의 정수여야 합니다.")

    image_extension = image_extension.lower().replace(".", "")
    if image_extension not in {"png", "jpg", "jpeg"}:
        raise ValueError("image_extension은 png, jpg, jpeg 중 하나여야 합니다.")

    if output_dir is None:
        save_dir = video_path.parent / f"{video_path.stem}_frames_every_{frame_number}"
    else:
        save_dir = Path(output_dir)

    save_dir.mkdir(parents=True, exist_ok=True)

    reader = imageio.get_reader(str(video_path))
    processed_count = 0
    saved_count = 0

    try:
        for idx, frame in enumerate(reader):
            processed_count += 1

            if idx % frame_number != 0:
                continue

            image = prepare_image(frame)
            save_path = save_dir / f"frame_{idx:06d}.{image_extension}"

            if image_extension in {"jpg", "jpeg"} and image.mode == "RGBA":
                image = image.convert("RGB")

            image.save(save_path)
            saved_count += 1

            if saved_count % 100 == 0:
                print(f"저장 완료: {saved_count}장 (현재 프레임 {idx})")

    finally:
        reader.close()

    print("=" * 50)
    print(f"입력 파일: {video_path}")
    print(f"전체 확인 프레임 수: {processed_count}")
    print(f"저장된 이미지 수: {saved_count}")
    print(f"저장 폴더: {save_dir}")
    print("완료")


save_video_frames(
    file_path=file_path,
    frame_number=frame_number,
    image_extension=image_extension,
    output_dir=output_dir,
)


C:\Users\user\AppData\Local\Temp\ipykernel_25768\4247688622.py:27: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(frame, mode="RGB")


저장 완료: 100장 (현재 프레임 99)
저장 완료: 200장 (현재 프레임 199)
저장 완료: 300장 (현재 프레임 299)
저장 완료: 400장 (현재 프레임 399)
저장 완료: 500장 (현재 프레임 499)
저장 완료: 600장 (현재 프레임 599)
저장 완료: 700장 (현재 프레임 699)
저장 완료: 800장 (현재 프레임 799)
저장 완료: 900장 (현재 프레임 899)
저장 완료: 1000장 (현재 프레임 999)
입력 파일: C:\Users\user\Desktop\Gibeom\1. HI Lab\0. Projects\0. On going\1. [Lead] ionic DC-TENG\0. 실험자료\8. Gibeom touch\260604_지원 카톡 합본 (최종)\DC Final  movie.gif
전체 확인 프레임 수: 1000
저장된 이미지 수: 1000
저장 폴더: C:\Users\user\Desktop\Gibeom\1. HI Lab\0. Projects\0. On going\1. [Lead] ionic DC-TENG\0. 실험자료\8. Gibeom touch\260604_지원 카톡 합본 (최종)\DC Final  movie_frames_every_1
완료
